### SkyScout: Fine-Tuning and Benchmarking YOLO and Transformer Detectors for Small-Object Detection in Drone Imagery

Francis Neil Moral, TensorFlow II project, Fanshawe College

This notebook runs the full project. The code is in `skyscout.py` and every setting is in `config.yaml`.
Each cell can be run again after a disconnect: finished steps are skipped, training continues from the last saved epoch
and evaluation continues from the last saved image. Train and eval times are recorded in `results/STATUS.md`.

Only want to try the trained models? Run **1. Setup**, then jump to **11. Demo**.

#### 1. Setup

Finds the machine type (Colab, server or local PC) by itself.

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/fnmoral/skyscout-small-object-detection-yolo-vs-dfine-visdrone.git'
IN_COLAB = 'google.colab' in sys.modules or 'COLAB_RELEASE_TAG' in os.environ
# Recommended: set to False when Drive cannot be mounted (for example a Colab runtime used from VS Code)
USE_DRIVE = True

if IN_COLAB:
    PROJECT_DIR = '/content/SkyScout'
    if USE_DRIVE:
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            # Recommended: keep the project on Drive so checkpoints and models survive a disconnect
            PROJECT_DIR = '/content/drive/MyDrive/SkyScout'
        except Exception as e:
            print(f'Drive not mounted ({type(e).__name__}), using {PROJECT_DIR} for this session only')
    if not os.path.exists(os.path.join(PROJECT_DIR, 'skyscout.py')):
        subprocess.run(['git', 'clone', REPO_URL, PROJECT_DIR], check=True)
    elif os.path.exists(os.path.join(PROJECT_DIR, '.git')):
        subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--ff-only'], check=False)
    os.chdir(PROJECT_DIR)
    # Recommended: if this upgrades TensorFlow, restart the session once and run this cell again
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
sys.path.insert(0, os.getcwd())

from IPython.display import Image, Markdown, display
import importlib

# running this cell again: stop an old demo and load the new code
if 'skyscout' in sys.modules:
    for old_app in sys.modules['skyscout']._DEMO:
        old_app.close()
    sk = importlib.reload(sys.modules['skyscout'])
else:
    import skyscout as sk

version = subprocess.run(['git', 'log', '-1', '--format=%h %cd', '--date=short'], capture_output=True, text=True).stdout.strip()
print('Code version:', version, '| Profile:', sk.PROFILE, '| GPU:', sk.gpu_name())
sk.show_status()

#### 2. Dataset

VisDrone2019-DET is downloaded once into `downloads/`. Ignored regions and the "others" class are dropped.

In [ ]:
sk.prepare_dataset()
sk.dataset_summary()

#### 3. Time one epoch

Estimate the training time before starting a full run. Results go to `results/timing.json` and are not measured twice.

In [ ]:
sk.time_epoch('yolo11s_640')
sk.time_epoch('dfine_full_640')

#### 4. Experiment 1: YOLO11s baseline at 640

In [ ]:
sk.run_experiment('yolo11s_640')
display(Markdown(sk.eval_dir('yolo11s_640', 'test').joinpath('report.md').read_text()))

#### 5. Experiment 2: YOLO26s at 640

In [ ]:
sk.run_experiment('yolo26s_640')
display(Markdown(sk.eval_dir('yolo26s_640', 'test').joinpath('report.md').read_text()))

#### 6. Experiment 3: D-FINE fine-tuning setups

Three setups, 30 epochs each: frozen backbone, full fine-tune, and full fine-tune with layer-wise learning rate decay.
The best one (by val mAP50-95) is then trained longer.

In [ ]:
for name in sk.CFG['dfine_setup_compare']:
    sk.run_experiment(name)
print('Best setup:', sk.best_dfine())

In [ ]:
sk.run_experiment('dfine_best_long_640')
display(Markdown(sk.eval_dir('dfine_best_long_640', 'test').joinpath('report.md').read_text()))

#### 7. Experiment 4: input size 640 vs 1024

The 1024 runs are set to the server profile in `config.yaml`. On other machines these cells only print a recommendation.

In [ ]:
for name in ['yolo11s_1024', 'yolo26s_1024', 'dfine_best_1024']:
    sk.run_experiment(name)

#### 8. Experiment 5: SAHI tiled inference

In [ ]:
sk.run_sahi()

#### 9. Experiment 6: data efficiency (D-FINE on 25%, 50%, 100% of the training set)

In [ ]:
for name in ['dfine_best_25', 'dfine_best_50']:
    sk.run_experiment(name)

#### 10. Results

Latency is re-measured for every saved model on this one machine, so all speeds come from the same GPU.

In [ ]:
sk.benchmark_latency()
sk.make_report()
display(Markdown(sk.RESULTS.joinpath('comparison.md').read_text().split('![')[0]))
for p in sorted(sk.PLOTS.glob('*.png')):
    display(Markdown(f'**{p.stem}**'), Image(str(p)))
sk.show_status()

#### 11. Demo

For trying the models out (the scores were already measured above). Pick a model, then run it on a drone photo or video
and get the boxes, a count per class and the speed. A sample clip from the VisDrone test-dev set is made the first time.
On Colab a public link is printed.

Only want to try the demo? Run 1. Setup, then this section. The trained models come from `models/` (copy them there, for example on Drive) or are downloaded from `demo.models_url`. Sample images and a sample clip are in `samples/`, so the dataset is not needed.

In [ ]:
# Unzips the trained models into models/ if they are missing: from demo.models_url, or pass a path,
# for example sk.fetch_models('/content/drive/MyDrive/skyscout_models.zip')
sk.fetch_models()
sk.launch_demo()